# IIoT Intrusion Detection System — N-BaIoT (Multi-Class)

**Models:** Centralized CNN-GRU · Autoencoder · Federated CNN-GRU (FedAvg)  
**Dataset:** N-BaIoT  
**Classes:** 11 (benign + 5 Mirai + 5 Gafgyt)

In [ ]:
import os, random, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_auc_score
)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv1D, GRU, Dense, Dropout, Input, MaxPooling1D, Flatten
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)

## 1. Load Dataset

In [ ]:
_NOISE_TOKENS   = {'attacks', 'traffic', 'attack', ''}
_KNOWN_FAMILIES = {'mirai', 'gafgyt'}

def get_attack_subtype(fname):
    stem  = fname.lower().replace('.csv', '')
    parts = [p for p in stem.replace('.', '_').split('_')
             if p not in _NOISE_TOKENS and not p.isdigit()]
    if not parts:
        return None
    if parts[0] == 'benign':
        return 'benign'
    if parts[0] in _KNOWN_FAMILIES and len(parts) >= 2:
        return f'{parts[0]}_{parts[-1]}'
    return None

ALLOWED_CLASSES = {
    'benign',
    'mirai_ack', 'mirai_syn', 'mirai_udp', 'mirai_udpplain', 'mirai_scan',
    'gafgyt_tcp', 'gafgyt_udp', 'gafgyt_junk', 'gafgyt_combo', 'gafgyt_scan',
}

MAX_PER_CLASS = 100_000

def find_dataset_path(base='/kaggle/input', keywords=('nbaiot', 'n-baiot', 'n_baiot')):
    for root, dirs, files in os.walk(base):
        if any(f.endswith('.csv') for f in files):
            if any(k in root.lower() for k in keywords):
                return root
    return None

DATA_PATH = find_dataset_path()
if DATA_PATH is None:
    raise FileNotFoundError("N-BaIoT dataset not found.")

csv_files = [f for f in os.listdir(DATA_PATH)
             if f.endswith('.csv')
             and f not in ('data_summary.csv', 'features.csv', 'device_info.csv')]

# Track how many rows we've collected per class to enforce the cap during loading
class_counts = {c: 0 for c in ALLOWED_CLASSES}
df_list, skipped = [], []

for f in csv_files:
    label = get_attack_subtype(f)
    if label is None or label not in ALLOWED_CLASSES:
        skipped.append(f)
        continue

    remaining = MAX_PER_CLASS - class_counts[label]
    if remaining <= 0:
        continue

    df = pd.read_csv(os.path.join(DATA_PATH, f))
    prefix    = f.lower().split('.')[0].split('_')[0]
    device_id = int(prefix) if prefix.isdigit() else 0

    if len(df) > remaining:
        df = df.sample(remaining, random_state=42)

    df['label']     = label
    df['device_id'] = device_id
    class_counts[label] += len(df)
    df_list.append(df)

data = pd.concat(df_list, ignore_index=True)
del df_list

print(f'Classes: {len(data["label"].unique())}, Shape: {data.shape}')
print(data['label'].value_counts().sort_index())

## 2. Preprocessing

In [ ]:
# Augmentation — inject noise into MI_dir_L5_weight
if 'MI_dir_L5_weight' in data.columns:
    data['MI_dir_L5_weight'] += np.random.normal(0, 1, len(data))

# Drop highly correlated features (compute on a sample to save memory)
numeric_cols = [c for c in data.select_dtypes(include=[np.number]).columns
                if c not in ('label', 'device_id')]
sample = data[numeric_cols].sample(min(len(data), 100_000), random_state=42)
corr = sample.corr().abs()
del sample
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_cols = [c for c in upper.columns if any(upper[c] > 0.98)]
data = data.drop(columns=drop_cols, errors='ignore')
print(f'Dropped {len(drop_cols)} correlated columns → {data.shape}')

# Shuffle
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Encode labels
device_ids = data['device_id'].values.astype(np.int32)
le = LabelEncoder()
y_int = le.fit_transform(data['label'].values)
NUM_CLASSES = len(le.classes_)
print(f'Classes ({NUM_CLASSES}): {list(le.classes_)}')

feature_cols = [c for c in data.select_dtypes(include=[np.number]).columns
                if c not in ('label', 'device_id')]
X = data[feature_cols].fillna(0).values.astype(np.float32)
del data

# Train / test split
X_train, X_test, y_train, y_test, dev_train, dev_test = train_test_split(
    X, y_int, device_ids, test_size=0.2, random_state=42, stratify=y_int)
del X

X_test_raw = X_test.copy()

# Standardise
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

# One-hot encode
y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_test_cat = to_categorical(y_test, num_classes=NUM_CLASSES)

# Reshape for Conv1D
X_train_r = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_r = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
INPUT_SHAPE = (X_train_r.shape[1], 1)

print(f'X_train: {X_train_r.shape}  X_test: {X_test_r.shape}')

## 3. Centralized CNN-GRU (AttackNet)

In [ ]:
def build_attacknet(input_shape, num_classes):
    m = Sequential([
        Conv1D(64, kernel_size=5, activation='relu', padding='same', input_shape=input_shape),
        Conv1D(32, kernel_size=5, activation='relu', padding='same'),
        MaxPooling1D(pool_size=4),
        GRU(32, return_sequences=True, activation='relu'),
        GRU(16, return_sequences=True),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dropout(0.1),
        Dense(num_classes, activation='softmax')
    ])
    m.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy', metrics=['accuracy'])
    return m

model = build_attacknet(INPUT_SHAPE, NUM_CLASSES)
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

history = model.fit(
    X_train_r, y_train_cat,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

### Evaluation

In [ ]:
y_pred_prob = model.predict(X_test_r)
y_pred = np.argmax(y_pred_prob, axis=1)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
rec  = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1   = f1_score(y_test, y_pred, average='macro', zero_division=0)
auc  = roc_auc_score(y_test_cat, y_pred_prob, multi_class='ovr', average='macro')

print('Centralized CNN-GRU Results')
print('-' * 45)
print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC  : {auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

## 4. Autoencoder — Anomaly Detection

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Multi-Class CNN-GRU (Centralized)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

# Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].set_ylim([0.85, 1.01])

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()

# Metrics Bar Chart
metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': auc}
plt.bar(metrics.keys(), metrics.values(), color='steelblue', edgecolor='white')
plt.ylim(0.8, 1.0); plt.ylabel('Score')
plt.title('Centralized CNN-GRU Metrics')
plt.tight_layout()
plt.savefig('/kaggle/working/metrics.png', dpi=150)
plt.show()

## 5. Federated Learning — CNN-GRU (FedAvg)

In [ ]:
BENIGN_IDX = list(le.classes_).index('benign')
X_benign = X_train[y_train == BENIGN_IDX]
print(f'Benign training samples: {X_benign.shape}')

input_dim = X_benign.shape[1]
ae_input  = Input(shape=(input_dim,))
enc = Dense(64, activation='relu')(ae_input)
enc = Dense(32, activation='relu')(enc)
dec = Dense(64, activation='relu')(enc)
dec = Dense(input_dim, activation='linear')(dec)

autoencoder = Model(inputs=ae_input, outputs=dec)
autoencoder.compile(optimizer='adam', loss='mse')

ae_callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

autoencoder.fit(X_benign, X_benign, epochs=20, batch_size=256,
                validation_split=0.1, callbacks=ae_callbacks, verbose=1)

# Threshold = 95th percentile of benign reconstruction error
benign_recon = autoencoder.predict(X_benign)
benign_error = np.mean(np.square(X_benign - benign_recon), axis=1)
threshold = np.percentile(benign_error, 95)
print(f'Anomaly threshold: {threshold:.6f}')

# Evaluate
X_test_recon = autoencoder.predict(X_test)
recon_error = np.mean(np.square(X_test - X_test_recon), axis=1)
y_ae_pred = (recon_error > threshold).astype(int)
y_ae_true = (y_test != BENIGN_IDX).astype(int)

ae_acc  = accuracy_score(y_ae_true, y_ae_pred)
ae_prec = precision_score(y_ae_true, y_ae_pred, zero_division=0)
ae_rec  = recall_score(y_ae_true, y_ae_pred, zero_division=0)
ae_f1   = f1_score(y_ae_true, y_ae_pred, zero_division=0)

print(f'\nAutoencoder Results (Benign vs Attack)')
print('-' * 45)
print(f'Accuracy : {ae_acc:.4f}')
print(f'Precision: {ae_prec:.4f}')
print(f'Recall   : {ae_rec:.4f}')
print(f'F1-score : {ae_f1:.4f}')

# Reconstruction Error Distribution
plt.figure(figsize=(10, 5))
plt.hist(recon_error[y_ae_true == 0], bins=50, alpha=0.6, label='Benign', color='green')
plt.hist(recon_error[y_ae_true == 1], bins=50, alpha=0.6, label='Attack', color='red')
plt.axvline(threshold, color='black', linestyle='--', linewidth=2,
            label=f'Threshold = {threshold:.4f}')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Frequency')
plt.title('Autoencoder Reconstruction Error Distribution')
plt.legend(); plt.tight_layout()
plt.savefig('/kaggle/working/ae_reconstruction_error.png', dpi=150)
plt.show()

## 6. Save Artifacts

In [ ]:
# Partition by device (non-IID)
unique_devices = np.unique(dev_train)
NUM_CLIENTS = len(unique_devices)

client_X, client_y = [], []
for dev_id in unique_devices:
    mask = dev_train == dev_id
    client_X.append(X_train_r[mask])
    client_y.append(y_train_cat[mask])

print(f'Federated clients: {NUM_CLIENTS}')
for i, dev_id in enumerate(unique_devices):
    labels = np.argmax(client_y[i], axis=1)
    u, c = np.unique(labels, return_counts=True)
    print(f'  Client {i+1} (Device {dev_id}): {client_X[i].shape}  '
          f'{dict(zip([le.classes_[k] for k in u], c.tolist()))}')

# FedAvg training
GLOBAL_ROUNDS, LOCAL_EPOCHS = 5, 3
client_sizes = [len(client_X[i]) for i in range(NUM_CLIENTS)]
total_samples = sum(client_sizes)

global_model = build_attacknet(INPUT_SHAPE, NUM_CLASSES)

for rnd in range(GLOBAL_ROUNDS):
    print(f'\nRound {rnd+1}/{GLOBAL_ROUNDS}')
    client_weights = []
    for i in range(NUM_CLIENTS):
        local = build_attacknet(INPUT_SHAPE, NUM_CLASSES)
        local.set_weights(global_model.get_weights())
        local.fit(client_X[i], client_y[i], epochs=LOCAL_EPOCHS, batch_size=32, verbose=0)
        client_weights.append(local.get_weights())
        print(f'  Client {i+1} trained ({client_sizes[i]} samples)')

    avg_weights = [
        sum((client_sizes[i] / total_samples) * client_weights[i][layer]
            for i in range(NUM_CLIENTS))
        for layer in range(len(client_weights[0]))
    ]
    global_model.set_weights(avg_weights)
    print('  Global model updated')

# Evaluate
fl_prob = global_model.predict(X_test_r, verbose=0)
fl_pred = np.argmax(fl_prob, axis=1)

fl_acc  = accuracy_score(y_test, fl_pred)
fl_prec = precision_score(y_test, fl_pred, average='macro', zero_division=0)
fl_rec  = recall_score(y_test, fl_pred, average='macro', zero_division=0)
fl_f1   = f1_score(y_test, fl_pred, average='macro', zero_division=0)
fl_auc  = roc_auc_score(y_test_cat, fl_prob, multi_class='ovr', average='macro')

print('\nFederated CNN-GRU Results')
print('-' * 45)
print(f'Accuracy : {fl_acc:.4f}')
print(f'Precision: {fl_prec:.4f}')
print(f'Recall   : {fl_rec:.4f}')
print(f'F1-score : {fl_f1:.4f}')
print(f'ROC-AUC  : {fl_auc:.4f}')
print()
print(classification_report(y_test, fl_pred, target_names=le.classes_, zero_division=0))

# Centralized vs Federated comparison
x_pos = np.arange(2)
w = 0.3
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x_pos - w/2, [acc, fl_acc], w, label='Accuracy', color='steelblue')
ax.bar(x_pos + w/2, [f1, fl_f1], w, label='F1 (macro)', color='coral')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Centralized', 'Federated'])
ax.set_ylim(0.85, 1.0); ax.set_ylabel('Score')
ax.set_title('Centralized vs Federated CNN-GRU')
ax.legend(); plt.tight_layout()
plt.savefig('/kaggle/working/centralized_vs_federated.png', dpi=150)
plt.show()

In [ ]:
OUTPUT_DIR = '/kaggle/working'

model.save(os.path.join(OUTPUT_DIR, 'model_mc.keras'))
global_model.save(os.path.join(OUTPUT_DIR, 'model_mc_fl.keras'))
autoencoder.save(os.path.join(OUTPUT_DIR, 'autoencoder_ids_mc.keras'))

with open(os.path.join(OUTPUT_DIR, 'scaler_mc.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
with open(os.path.join(OUTPUT_DIR, 'le_mc.pkl'), 'wb') as f:
    pickle.dump(le, f)
with open(os.path.join(OUTPUT_DIR, 'ae_threshold_mc.pkl'), 'wb') as f:
    pickle.dump(float(threshold), f)

label_map = {i: c for i, c in enumerate(le.classes_)}
with open(os.path.join(OUTPUT_DIR, 'label_map_mc.pkl'), 'wb') as f:
    pickle.dump(label_map, f)

test_export = pd.DataFrame(X_test_raw, columns=feature_cols)
test_export['label'] = le.inverse_transform(y_test)
test_export.to_csv(os.path.join(OUTPUT_DIR, 'test_samples_mc.csv'), index=False)

print('All artifacts saved to /kaggle/working/')